In [0]:
# Databricks widget parameter
dbutils.widgets.text("volume_directory", "/Volumes/usa/osint/rss/medpage/", "Volume Directory")
output_dir = dbutils.widgets.get("volume_directory")

In [0]:
"""
MedPage Today RSS Feed Downloader
----------------------------------
Scrapes the RSS feed listing page from MedPage Today,
extracts all RSS feed URLs, downloads each feed, and
saves the XML content to the specified volume directory.

Parameters:
    volume_directory - Target volume path (e.g. /Volumes/usa/osint/rss/medpage/)
"""

import requests
from bs4 import BeautifulSoup
import os
import re
import http.cookiejar
import urllib.request
from urllib.parse import urlparse
from datetime import datetime

# Configuration
LISTING_URL = "https://support.medpagetoday.com/hc/en-us/articles/206269823-RSS-Feeds"
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Cache-Control": "max-age=0",
}

# Datestamp prefix for saved files (YYYYMMDD format)
DATE_PREFIX = datetime.now().strftime("%Y%m%d")


def get_rss_feed_urls(listing_url):
    """Fetch the RSS listing page and extract all RSS feed URLs."""
    print(f"Fetching RSS feed listing from: {listing_url}")

    # Use urllib with cookie handling to bypass bot detection on Zendesk pages
    cookie_jar = http.cookiejar.CookieJar()
    opener = urllib.request.build_opener(
        urllib.request.HTTPCookieProcessor(cookie_jar),
        urllib.request.HTTPRedirectHandler(),
    )
    req = urllib.request.Request(listing_url)
    req.add_header("User-Agent", USER_AGENT)
    req.add_header("Accept", "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8")
    req.add_header("Accept-Language", "en-US,en;q=0.9")
    req.add_header("Upgrade-Insecure-Requests", "1")
    req.add_header("Sec-Fetch-Dest", "document")
    req.add_header("Sec-Fetch-Mode", "navigate")
    req.add_header("Sec-Fetch-Site", "none")
    req.add_header("Sec-Fetch-User", "?1")

    response = opener.open(req, timeout=30)
    html = response.read().decode("utf-8", errors="replace")

    soup = BeautifulSoup(html, "html.parser")

    # Find all links that look like RSS feeds (typically ending in .xml or /rss or containing 'feed')
    rss_urls = set()
    for link in soup.find_all("a", href=True):
        href = link["href"]
        if any(keyword in href.lower() for keyword in ["rss", "feed", ".xml", "atom"]):
            # Ensure absolute URL
            if href.startswith("/"):
                href = f"https://www.medpagetoday.com{href}"
            if href.startswith("http"):
                rss_urls.add(href)

    print(f"Found {len(rss_urls)} RSS feed URLs")
    return sorted(rss_urls)


def sanitize_filename(url):
    """Generate a safe filename from a URL, prefixed with today's datestamp."""
    parsed = urlparse(url)
    # Use the path to create a meaningful filename
    path_parts = [p for p in parsed.path.strip("/").split("/") if p]
    if path_parts:
        name = "_".join(path_parts[-2:]) if len(path_parts) > 1 else path_parts[-1]
    else:
        name = parsed.netloc.replace(".", "_")
    # Remove unsafe characters
    name = re.sub(r'[^\w\-.]', '_', name)
    if not name.endswith(".xml"):
        name += ".xml"
    # Prefix with datestamp
    return f"{DATE_PREFIX}_{name}"


def download_feed(url, output_dir):
    """Download a single RSS feed and save it to the output directory."""
    filename = sanitize_filename(url)
    filepath = os.path.join(output_dir, filename)

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(response.text)

        print(f"  \u2713 Saved: {filename} ({len(response.text):,} bytes)")
        return True
    except requests.exceptions.RequestException as e:
        print(f"  \u2717 Failed: {filename} - {e}")
        return False


print("=" * 60)
print("MedPage Today RSS Feed Downloader")
print(f"Timestamp: {datetime.now().isoformat()}")
print("=" * 60)

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)
print(f"\nOutput directory: {output_dir}")
print(f"File prefix: {DATE_PREFIX}_")

# Step 1: Get all RSS feed URLs from the listing page
rss_urls = get_rss_feed_urls(LISTING_URL)

if not rss_urls:
    print("\nNo RSS feed URLs found. The page structure may have changed.")
    print("Trying common MedPage Today RSS patterns...")
    # Fallback: known MedPage Today RSS feed patterns
    rss_urls = [
        "https://www.medpagetoday.com/rss/headlines.xml",
        "https://www.medpagetoday.com/rss/cardiology.xml",
        "https://www.medpagetoday.com/rss/oncology.xml",
        "https://www.medpagetoday.com/rss/neurology.xml",
        "https://www.medpagetoday.com/rss/infectiousdisease.xml",
        "https://www.medpagetoday.com/rss/endocrinology.xml",
        "https://www.medpagetoday.com/rss/pulmonology.xml",
        "https://www.medpagetoday.com/rss/rheumatology.xml",
        "https://www.medpagetoday.com/rss/psychiatry.xml",
        "https://www.medpagetoday.com/rss/dermatology.xml",
    ]
    print(f"Using {len(rss_urls)} fallback URLs")

# Step 2: Download each RSS feed
print(f"\nDownloading {len(rss_urls)} feeds...\n")
success_count = 0
fail_count = 0

for url in rss_urls:
    if download_feed(url, output_dir):
        success_count += 1
    else:
        fail_count += 1

# Summary
print("\n" + "=" * 60)
print(f"Download complete: {success_count} succeeded, {fail_count} failed")
print(f"Files saved to: {output_dir}")
print("=" * 60)